# MassSpecGym v1.5 consistency check — TSV / MGF vs original PubChem-standardised release

The v1.5 release re-canonicalises the `smiles` column of `MassSpecGym.tsv`
(originally PubChem-standardised) into RDKit canonical + stereo-stripped form
and recomputes the dependent `formula` / `inchikey` columns from the new SMILES
(so the JSON join key matches). Everything else — spectra metadata, peak lists,
folds — should be identical row-by-row.

The TSV is the canonical source: `MassSpecGym1.5.mgf` is then a one-pass
derivation of `MassSpecGym1.5.tsv` via `matchms.exporting.save_as_mgf`
(script `MassSpecGym/scripts/fixes/build_mgf_from_tsv.py`).

This notebook performs the check against the **PubChem-standardised originals**
(`MassSpecGym.tsv`, `MassSpecGym.mgf`):

1. TSV: align on `identifier`, report per-column diff counts.
2. MGF: align on `IDENTIFIER`, report per-header diff counts and verify
   peak-list equality.

In [1]:
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd

DATA = Path('/pfs/lustrep2/scratch/project_465002061/rbushuie/DreaMS-Mol_dev/MassSpecGym/data')

V15_TSV = DATA / 'v1.5/MassSpecGym1.5.tsv'
V15_MGF = DATA / 'v1.5/MassSpecGym1.5.mgf'
# Original PubChem-standardised release (pre-v1.5).
ORIG_TSV = DATA / 'MassSpecGym.tsv'
ORIG_MGF = DATA / 'MassSpecGym.mgf'

EXPECTED_CHANGED_COLS = {'smiles', 'formula', 'inchikey'}  # all driven by canonicalisation/stripping
EXPECTED_CHANGED_MGF = {'SMILES', 'FORMULA', 'INCHIKEY'}

## 1 — TSV: v1.5 vs original

Load both, align on `identifier`, and count per-column differences.

In [2]:
v15 = pd.read_csv(V15_TSV, sep='\t')
orig = pd.read_csv(ORIG_TSV, sep='\t')
print(f'v1.5    TSV: {len(v15):,} rows × {v15.shape[1]} cols')
print(f'original TSV: {len(orig):,} rows × {orig.shape[1]} cols')
assert set(v15.columns) == set(orig.columns), 'column sets differ'
assert set(v15['identifier']) == set(orig['identifier']), 'identifier sets differ'

merged = orig.merge(v15, on='identifier', suffixes=('_orig', '_v15'))
merged = merged.reindex(sorted(merged.columns), axis=1)

n = len(merged)
rows = []
for col in [c for c in orig.columns if c != 'identifier']:
    a = merged[f'{col}_orig']
    b = merged[f'{col}_v15']
    if a.dtype.kind in 'fc':  # float / complex
        diff = ~np.isclose(a.fillna(-1e30), b.fillna(-1e30), atol=1e-3, rtol=0)
    else:
        diff = (a.astype(object).where(a.notna(), '<NA>') != b.astype(object).where(b.notna(), '<NA>'))
    n_diff = int(diff.sum())
    rows.append({'column': col, 'n_diff': n_diff, 'pct_diff': round(n_diff / n * 100, 3),
                 'expected_to_change': col in EXPECTED_CHANGED_COLS})
tsv_diff_df = pd.DataFrame(rows).set_index('column')
display(tsv_diff_df)

unexpected = tsv_diff_df[(tsv_diff_df['n_diff'] > 0) & ~tsv_diff_df['expected_to_change']]
if not unexpected.empty:
    print('\nUNEXPECTED diffs (columns we did NOT plan to touch):')
    display(unexpected)
else:
    print('\nAll diffs are confined to the expected columns (smiles, formula, inchikey).')

v1.5    TSV: 231,104 rows × 14 cols
original TSV: 231,104 rows × 14 cols


,n_diff,pct_diff,expected_to_change
column,,,
mzs,0,0.00,False
intensities,0,0.00,False
smiles,225281,97.48,True
inchikey,0,0.00,True
formula,0,0.00,True
precursor_formula,0,0.00,False
parent_mass,0,0.00,False
precursor_mz,0,0.00,False
adduct,0,0.00,False



All diffs are confined to the expected columns (smiles, formula, inchikey).


### 1a — Examples of the changed `smiles` / `formula` / `inchikey` cells

In [3]:
for col in ['smiles', 'formula', 'inchikey']:
    diff_mask = merged[f'{col}_orig'].astype(object).where(merged[f'{col}_orig'].notna(), '<NA>') != \
                merged[f'{col}_v15'].astype(object).where(merged[f'{col}_v15'].notna(), '<NA>')
    diff_rows = merged.loc[diff_mask, ['identifier', f'{col}_orig', f'{col}_v15']].head(6)
    print(f'--- {col} — first 6 diffs ({int(diff_mask.sum()):,} total) ---')
    display(diff_rows)

--- smiles — first 6 diffs (225,281 total) ---


,identifier,smiles_orig,smiles_v15
0,MassSpecGymID0000001,CC(=O)N[C@@H](CC1=CC=CC=C1)C2=CC(=CC(=O)O2)OC,COc1cc(C(Cc2ccccc2)NC(C)=O)oc(=O)c1
1,MassSpecGymID0000002,CC(=O)N[C@@H](CC1=CC=CC=C1)C2=CC(=CC(=O)O2)OC,COc1cc(C(Cc2ccccc2)NC(C)=O)oc(=O)c1
2,MassSpecGymID0000003,CC(=O)N[C@@H](CC1=CC=CC=C1)C2=CC(=CC(=O)O2)OC,COc1cc(C(Cc2ccccc2)NC(C)=O)oc(=O)c1
3,MassSpecGymID0000004,CC(=O)N[C@@H](CC1=CC=CC=C1)C2=CC(=CC(=O)O2)OC,COc1cc(C(Cc2ccccc2)NC(C)=O)oc(=O)c1
4,MassSpecGymID0000005,CC(=O)N[C@@H](CC1=CC=CC=C1)C2=CC(=CC(=O)O2)OC,COc1cc(C(Cc2ccccc2)NC(C)=O)oc(=O)c1
5,MassSpecGymID0000006,CC(=O)N[C@@H](CC1=CC=CC=C1)C2=CC(=CC(=O)O2)OC,COc1cc(C(Cc2ccccc2)NC(C)=O)oc(=O)c1


--- formula — first 6 diffs (0 total) ---


,identifier,formula_orig,formula_v15


--- inchikey — first 6 diffs (0 total) ---


,identifier,inchikey_orig,inchikey_v15


## 2 — MGF: v1.5 vs original

Parse each MGF block by block, align on `IDENTIFIER`, count per-header
differences and verify the peak lists are byte-identical.

In [4]:
def parse_mgf(path: Path):
    """Yield (identifier, headers: dict, peaks: list[str])."""
    headers = None
    peaks = None
    with path.open() as f:
        for line in f:
            line = line.rstrip('\n')
            if line == 'BEGIN IONS':
                headers, peaks = {}, []
                continue
            if line == 'END IONS':
                yield headers.get('IDENTIFIER'), headers, peaks
                headers, peaks = None, None
                continue
            if headers is None:
                continue
            if '=' in line and line[0].isalpha():
                k, _, v = line.partition('=')
                headers[k.strip()] = v.strip()
            elif line:
                peaks.append(line.strip())

print(f'Loading {V15_MGF.name} ...')
v15_mgf = {ident: (h, p) for ident, h, p in parse_mgf(V15_MGF) if ident}
print(f'  {len(v15_mgf):,} blocks')

print(f'Loading {ORIG_MGF.name} ...')
orig_mgf = {ident: (h, p) for ident, h, p in parse_mgf(ORIG_MGF) if ident}
print(f'  {len(orig_mgf):,} blocks')

shared_ids = sorted(set(v15_mgf) & set(orig_mgf))
only_v15 = set(v15_mgf) - set(orig_mgf)
only_orig = set(orig_mgf) - set(v15_mgf)
print(f'\nshared IDs: {len(shared_ids):,}')
print(f'only in v1.5: {len(only_v15):,}')
print(f'only in original: {len(only_orig):,}')

Loading MassSpecGym1.5.mgf ...


  231,104 blocks
Loading MassSpecGym.mgf ...


  231,104 blocks



shared IDs: 231,104
only in v1.5: 0
only in original: 0


In [5]:
header_diff: dict[str, int] = defaultdict(int)
peak_mismatch = 0
all_header_keys: set[str] = set()
for ident in shared_ids:
    h_v, p_v = v15_mgf[ident]
    h_o, p_o = orig_mgf[ident]
    all_header_keys.update(h_v); all_header_keys.update(h_o)
    for k in set(h_v) | set(h_o):
        if h_v.get(k) != h_o.get(k):
            header_diff[k] += 1
    if p_v != p_o:
        peak_mismatch += 1

rows = []
for k in sorted(all_header_keys):
    nd = header_diff.get(k, 0)
    rows.append({'header': k, 'n_diff': nd, 'pct_diff': round(nd / len(shared_ids) * 100, 3),
                 'expected_to_change': k in EXPECTED_CHANGED_MGF})
mgf_diff_df = pd.DataFrame(rows).set_index('header')
display(mgf_diff_df)
print(f'\nPeak lists differ in {peak_mismatch:,} / {len(shared_ids):,} blocks '
      f'({peak_mismatch / max(len(shared_ids), 1) * 100:.3f} %)')

unexpected = mgf_diff_df[(mgf_diff_df['n_diff'] > 0) & ~mgf_diff_df['expected_to_change']]
if not unexpected.empty:
    print('\nUNEXPECTED MGF diffs (headers we did NOT plan to touch):')
    display(unexpected)
else:
    print('\nAll MGF diffs are confined to the expected headers (SMILES, FORMULA, INCHIKEY).')

,n_diff,pct_diff,expected_to_change
header,,,
ADDUCT,0,0.000,False
COLLISION_ENERGY,1274,0.551,False
FOLD,0,0.000,False
FORMULA,0,0.000,True
IDENTIFIER,0,0.000,False
INCHIKEY,0,0.000,True
INSTRUMENT_TYPE,0,0.000,False
PARENT_MASS,123,0.053,False
PRECURSOR_FORMULA,0,0.000,False



Peak lists differ in 0 / 231,104 blocks (0.000 %)

UNEXPECTED MGF diffs (headers we did NOT plan to touch):


,n_diff,pct_diff,expected_to_change
header,,,
COLLISION_ENERGY,1274,0.551,False
PARENT_MASS,123,0.053,False
PRECURSOR_MZ,94,0.041,False


### 2a — Examples of changed MGF SMILES / FORMULA / INCHIKEY headers

In [6]:
import itertools
for key in ['SMILES', 'FORMULA', 'INCHIKEY']:
    print(f'--- {key} — first 6 diffs ---')
    n_shown = 0
    for ident in shared_ids:
        h_v, _ = v15_mgf[ident]; h_o, _ = orig_mgf[ident]
        if h_v.get(key) != h_o.get(key):
            print(f'  {ident}: orig={h_o.get(key)!r}  ->  v15={h_v.get(key)!r}')
            n_shown += 1
            if n_shown >= 6: break
    print()

--- SMILES — first 6 diffs ---
  MassSpecGymID0000001: orig='CC(=O)N[C@@H](CC1=CC=CC=C1)C2=CC(=CC(=O)O2)OC'  ->  v15='COc1cc(C(Cc2ccccc2)NC(C)=O)oc(=O)c1'
  MassSpecGymID0000002: orig='CC(=O)N[C@@H](CC1=CC=CC=C1)C2=CC(=CC(=O)O2)OC'  ->  v15='COc1cc(C(Cc2ccccc2)NC(C)=O)oc(=O)c1'
  MassSpecGymID0000003: orig='CC(=O)N[C@@H](CC1=CC=CC=C1)C2=CC(=CC(=O)O2)OC'  ->  v15='COc1cc(C(Cc2ccccc2)NC(C)=O)oc(=O)c1'
  MassSpecGymID0000004: orig='CC(=O)N[C@@H](CC1=CC=CC=C1)C2=CC(=CC(=O)O2)OC'  ->  v15='COc1cc(C(Cc2ccccc2)NC(C)=O)oc(=O)c1'
  MassSpecGymID0000005: orig='CC(=O)N[C@@H](CC1=CC=CC=C1)C2=CC(=CC(=O)O2)OC'  ->  v15='COc1cc(C(Cc2ccccc2)NC(C)=O)oc(=O)c1'
  MassSpecGymID0000006: orig='CC(=O)N[C@@H](CC1=CC=CC=C1)C2=CC(=CC(=O)O2)OC'  ->  v15='COc1cc(C(Cc2ccccc2)NC(C)=O)oc(=O)c1'

--- FORMULA — first 6 diffs ---



--- INCHIKEY — first 6 diffs ---


### 2b — Numeric equality re-check for the 3 "unexpected" MGF headers

The COLLISION_ENERGY / PARENT_MASS / PRECURSOR_MZ string-diffs are pure
float-formatting round-off (e.g. `41.490019999999994` vs `41.49002`) introduced
by `matchms.exporting.save_as_mgf` writing the same numeric value with a
cleaner `repr`. The numeric values themselves are identical at every digit
that matters — confirmed below as the float-difference is exactly zero
(or < 1e-9, well below any analytical precision).

In [7]:
numeric_residual = {}
for key in ['COLLISION_ENERGY', 'PARENT_MASS', 'PRECURSOR_MZ']:
    residuals = []
    for ident in shared_ids:
        a = v15_mgf[ident][0].get(key); b = orig_mgf[ident][0].get(key)
        if a == b: continue
        try:
            residuals.append(abs(float(a) - float(b)))
        except (TypeError, ValueError):
            residuals.append(float('nan'))
    if residuals:
        arr = np.asarray(residuals)
        numeric_residual[key] = {
            'n_string_diff': len(arr),
            'max_abs_float_diff': float(np.nanmax(arr)),
            'all_below_1e-6': bool((arr < 1e-6).all()),
        }
    else:
        numeric_residual[key] = {'n_string_diff': 0,
                                 'max_abs_float_diff': 0.0,
                                 'all_below_1e-6': True}
display(pd.DataFrame(numeric_residual).T)

,n_string_diff,max_abs_float_diff,all_below_1e-6
COLLISION_ENERGY,1274,0.0,True
PARENT_MASS,123,0.0,True
PRECURSOR_MZ,94,0.0,True


## 3 — Summary

In [8]:
tsv_unexpected = tsv_diff_df[(tsv_diff_df['n_diff'] > 0) & ~tsv_diff_df['expected_to_change']]
mgf_unexpected_strings = mgf_diff_df[(mgf_diff_df['n_diff'] > 0) & ~mgf_diff_df['expected_to_change']]
mgf_unexpected_real = mgf_unexpected_strings[~mgf_unexpected_strings.index.isin(numeric_residual)]
mgf_numeric_max_residual = max((d['max_abs_float_diff'] for d in numeric_residual.values()), default=0.0)

summary = {
    'TSV rows v1.5':                                       len(v15),
    'TSV rows original':                                   len(orig),
    'TSV cols where v1.5 differs from original':           int((tsv_diff_df['n_diff'] > 0).sum()),
    'TSV diffs confined to expected cols':                 bool(tsv_unexpected.empty),
    'MGF blocks v1.5':                                     len(v15_mgf),
    'MGF blocks original':                                 len(orig_mgf),
    'MGF blocks with mismatched peak lists':               peak_mismatch,
    'MGF headers where v1.5 differs (string)':             int((mgf_diff_df['n_diff'] > 0).sum()),
    'MGF headers with float-only formatting diff':         int(len(numeric_residual)),
    'MGF max absolute float diff across those headers':    mgf_numeric_max_residual,
    'MGF diffs are SMILES/FORMULA/INCHIKEY + numeric reformat only': bool(mgf_unexpected_real.empty),
}
display(pd.DataFrame([summary]).T.rename(columns={0: 'value'}))

,value
TSV rows v1.5,231104
TSV rows original,231104
TSV cols where v1.5 differs from original,1
TSV diffs confined to expected cols,True
MGF blocks v1.5,231104
MGF blocks original,231104
MGF blocks with mismatched peak lists,0
MGF headers where v1.5 differs (string),4
MGF headers with float-only formatting diff,3
MGF max absolute float diff across those headers,0.0
